# BPT Comparison

This notebook makes BPT diagnostics and matching all-field boundary mosaics for both DIG-corrected and no-DIG catalogs. Outputs are written to `bpt_comparison/<mode>/`, not `PAPER_PLOTS`.

In [1]:
from pathlib import Path
import sys

_here = Path.cwd().resolve()
_repo_candidates = (_here, *_here.parents)
REPO_ROOT = next((p for p in _repo_candidates if (p / 'CATALOGS').is_dir() and (p / 'Boundary_maps').is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError(f'Could not locate PAPER1 repo root from {_here}')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from m33_pipeline.notebook_setup import prepare_notebook
REPO_ROOT = prepare_notebook(REPO_ROOT)
print(f'Notebook working directory set to: {REPO_ROOT}')


Notebook working directory set to: /Users/emmajarvis/Documents/SIGNALS/M33/PAPER1


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from astropy.io import fits
from astropy.wcs import WCS
from reproject import reproject_interp
from reproject.mosaicking import find_optimal_celestial_wcs
from skimage.measure import find_contours
import astropy.units as u
from astropy.coordinates import SkyCoord
from matplotlib.colors import LogNorm

plt.rcParams.update({
    'font.family': 'serif',
    'mathtext.fontset': 'cm',
    'axes.linewidth': 1.4,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
})

from m33_pipeline.reporting import load_wr_catalog, load_snr_catalog

from scipy.optimize import curve_fit


In [3]:
CATALOG_METHOD = 'summed_map'
DIG_MODES = ['dig_subtracted', 'no_dig']
FIELDS = ('NE', 'NW', 'F5', 'SE', 'SW', 'F7', 'F6', 'F8', 'F9')
HALPHA_ONLY_FIELDS = {'F9'}
SNR_CUT = 3.0
OUTPUT_ROOT = Path('bpt_comparison')
OUTPUT_ROOT.mkdir(exist_ok=True)

BOUNDARY_DIR = Path('Boundary_maps/Boundary_map_100pc')
MAP_ROOT = REPO_ROOT.parent / 'M33-Maps'
KINEMATIC_SOURCE = Path('PAPER_PLOTS/summed_map/dig_subtracted/tables/M33_emission_region_properties.csv')

FIELD_DIR = {field: MAP_ROOT / f'M33-{field}' for field in FIELDS}
HAOIII_PATTERN = 'M33-{FIELD}_SN3.LineMaps.map.Ha+OIII.1x1.amplitude.fits'
HA_PATTERN = 'M33{FIELD}-Haflux.fits'


wr_catalog = load_wr_catalog()
snr_catalog = load_snr_catalog()
pn_catalog_path = Path('CATALOGS/planetary_nebulae/M33_PNe_combined_deduplicated.csv')
pn_catalog = pd.read_csv(pn_catalog_path) if pn_catalog_path.exists() else pd.DataFrame()
print(f'Loaded compact-object catalogs: WR={len(wr_catalog)}, SNR={len(snr_catalog)}, PN={len(pn_catalog)}')


Loaded compact-object catalogs: WR=33, SNR=163, PN=187


In [4]:
def mode_output_dir(mode, subdir=None):
    out = OUTPUT_ROOT / mode
    if subdir is not None:
        out = out / subdir
    out.mkdir(parents=True, exist_ok=True)
    return out


def catalog_path_for_mode(mode):
    base = Path('CATALOGS/flux_catalogs') / CATALOG_METHOD / mode
    for name in ['total_flux_catalog_combined.csv', 'total_flux_catalog.csv']:
        path = base / name
        if path.exists():
            return path
    raise FileNotFoundError(f'No combined catalog found in {base}')


def merge_kinematic_columns(df):
    if not KINEMATIC_SOURCE.exists():
        return df
    kin = pd.read_csv(KINEMATIC_SOURCE)
    rename = {'Field': 'field', 'ID': 'region_id'}
    kin = kin.rename(columns={k: v for k, v in rename.items() if k in kin.columns})
    key_cols = ['field', 'region_id']
    kin_cols = key_cols + [
        col for col in ['v_helio_mean_kms', 'v_helio_mean_err_kms', 'sigma_mean_kms', 'sigma_mean_err_kms', 'n_kinematic_spaxels']
        if col in kin.columns
    ]
    if not set(key_cols).issubset(df.columns) or not set(key_cols).issubset(kin.columns):
        return df
    value_cols = [col for col in kin_cols if col not in key_cols and col not in df.columns]
    if not value_cols:
        return df
    return df.merge(kin[key_cols + value_cols], on=key_cols, how='left', validate='many_to_one')


def load_catalog(mode, primary_only=True):
    path = catalog_path_for_mode(mode)
    df = pd.read_csv(path)
    if 'field' not in df.columns and 'Field' in df.columns:
        df['field'] = df['Field']
    if 'region_id' not in df.columns and 'ID' in df.columns:
        df['region_id'] = df['ID']
    df = merge_kinematic_columns(df)
    if primary_only and 'primary' in df.columns:
        df = df.loc[df['primary'].fillna(True)].copy().reset_index(drop=True)
    df.attrs['source_path'] = str(path)
    return add_bpt_columns(df)


def _numeric(df, col):
    if col not in df.columns:
        return np.full(len(df), np.nan, dtype=float)
    return pd.to_numeric(df[col], errors='coerce').to_numpy(dtype=float)



def classify_bpt_from_ratios(log_nii_ha, log_oiii_hb):
    x = np.asarray(log_nii_ha, dtype=float)
    y = np.asarray(log_oiii_hb, dtype=float)
    out = np.full(len(x), 'Unclassified', dtype=object)
    finite = np.isfinite(x) & np.isfinite(y)
    with np.errstate(divide='ignore', invalid='ignore'):
        kauffmann = 0.61 / (x - 0.05) + 1.3
        kewley = 0.61 / (x - 0.47) + 1.19
    sf = finite & (x < 0.05) & (y < kauffmann)
    composite = finite & ~sf & (x < 0.47) & (y < kewley)
    agn = finite & ~sf & ~composite
    out[sf] = 'Star-forming'
    out[composite] = 'Composite'
    out[agn] = 'AGN/Shock'
    return out



def add_nodig_dereddened_columns(df):
    # The DIG-corrected catalog stores pre-DIG fluxes as F_<line>_sum_nodig.
    # Reconstruct pre-DIG dereddened fluxes using each line's active dereddening factor.
    for line in ['Halpha', 'Hbeta', '[OIII]5007', '[NII]6583', '[SII]6716', '[SII]6731', '[OII]3727']:
        active = f'F_{line}_sum'
        active_dered = f'F_{line}_sum_dered'
        nodig = f'F_{line}_sum_nodig'
        nodig_dered = f'F_{line}_sum_dered_nodig'
        if nodig_dered in df.columns:
            continue
        if {active, active_dered, nodig}.issubset(df.columns):
            active_vals = pd.to_numeric(df[active], errors='coerce').to_numpy(dtype=float)
            dered_vals = pd.to_numeric(df[active_dered], errors='coerce').to_numpy(dtype=float)
            nodig_vals = pd.to_numeric(df[nodig], errors='coerce').to_numpy(dtype=float)
            with np.errstate(divide='ignore', invalid='ignore'):
                factor = np.where(np.isfinite(active_vals) & (active_vals > 0), dered_vals / active_vals, np.nan)
            df[nodig_dered] = nodig_vals * factor
    return df

def add_flux_ratio_columns(df, suffix, flux_suffix):
    ha = _numeric(df, f'F_Halpha_{flux_suffix}')
    hb = _numeric(df, f'F_Hbeta_{flux_suffix}')
    oiii = _numeric(df, f'F_[OIII]5007_{flux_suffix}')
    nii = _numeric(df, f'F_[NII]6583_{flux_suffix}')
    sii6716 = _numeric(df, f'F_[SII]6716_{flux_suffix}')
    sii6731 = _numeric(df, f'F_[SII]6731_{flux_suffix}')
    sii = sii6716 + sii6731
    with np.errstate(divide='ignore', invalid='ignore'):
        df[f'log_NII_Halpha_{suffix}'] = np.log10(nii / ha)
        df[f'log_SII_Halpha_{suffix}'] = np.log10(sii / ha)
        df[f'log_OIII_Hbeta_{suffix}'] = np.log10(oiii / hb)
        df[f'sii_halpha_ratio_{suffix}'] = sii / ha
    return df

def add_bpt_columns(df):
    df = df.copy()
    df = add_nodig_dereddened_columns(df)
    # Active columns are the current mode's fluxes. In dig_subtracted mode these are DIG-subtracted.
    df = add_flux_ratio_columns(df, 'plot', 'sum_dered')
    # The DIG-corrected catalog also carries pre-DIG fluxes as *_sum_nodig.
    if 'F_Halpha_sum_dered_nodig' in df.columns:
        df = add_flux_ratio_columns(df, 'nodig', 'sum_dered_nodig')
        df['BPT_class_nodig_from_flux'] = classify_bpt_from_ratios(
            df['log_NII_Halpha_nodig'].to_numpy(dtype=float),
            df['log_OIII_Hbeta_nodig'].to_numpy(dtype=float),
        )
    else:
        df['BPT_class_nodig_from_flux'] = df.get('BPT_class_sum_dered', pd.Series('Unclassified', index=df.index))

    df['low_snr_any_bpt_line'] = low_snr_any_line(df, snr_cut=SNR_CUT)
    df['high_snr_all_bpt_lines'] = ~df['low_snr_any_bpt_line']
    df['bpt_sf'] = df.get('BPT_class_sum_dered', pd.Series('', index=df.index)).astype(str).str.strip().eq('Star-forming')
    df['bpt_sf_nodig'] = pd.Series(df['BPT_class_nodig_from_flux'], index=df.index).astype(str).str.strip().eq('Star-forming')
    df['bpt_sf_high_snr'] = df['bpt_sf'] & df['high_snr_all_bpt_lines']
    df['nodig_bpt_sf_high_snr'] = df['bpt_sf_nodig'] & df['high_snr_all_bpt_lines']
    df['high_snr_non_bpt_sf'] = df['high_snr_all_bpt_lines'] & ~df['bpt_sf']
    df['nodig_sf_not_dig_sf_high_snr'] = df['nodig_bpt_sf_high_snr'] & ~df['bpt_sf']
    if 'radius_areaeq_pc' not in df.columns and 'radius_p50_pc' in df.columns:
        df['radius_areaeq_pc'] = df['radius_p50_pc']
    return df


def low_snr_any_line(df, snr_cut=SNR_CUT):
    snr_cols = [
        'SNR_Halpha_sum',
        'SNR_Hbeta_sum',
        'SNR_[OIII]5007_sum',
        'SNR_[NII]6583_sum',
        'SNR_[SII]6716_sum',
        'SNR_[SII]6731_sum',
    ]
    low = np.zeros(len(df), dtype=bool)
    for col in snr_cols:
        if col not in df.columns:
            low |= True
        else:
            vals = pd.to_numeric(df[col], errors='coerce').to_numpy(dtype=float)
            low |= ~np.isfinite(vals) | (vals < snr_cut)
    return low

catalogs = {mode: load_catalog(mode) for mode in DIG_MODES}
for mode, df in catalogs.items():
    print(f'{mode}: {len(df)} primary regions from {df.attrs["source_path"]}')
    print(f'  low S/N in any BPT line: {int(df["low_snr_any_bpt_line"].sum())}')
    print(f'  BPT SF and high S/N: {int(df["bpt_sf_high_snr"].sum())}')


dig_subtracted: 5902 primary regions from CATALOGS/flux_catalogs/summed_map/dig_subtracted/total_flux_catalog_combined.csv
  low S/N in any BPT line: 928
  BPT SF and high S/N: 4549
no_dig: 6397 primary regions from CATALOGS/flux_catalogs/summed_map/no_dig/total_flux_catalog_combined.csv
  low S/N in any BPT line: 1025
  BPT SF and high S/N: 4944


In [5]:
def bpt_limits(diagram):
    if diagram == 'nii':
        return (-2.0, 0.7), (-1.5, 1.6), r'log([NII]6583/H$\alpha$)'
    if diagram == 'sii':
        return (-1.6, 0.7), (-1.5, 1.6), r'log(([SII]6716+[SII]6731)/H$\alpha$)'
    raise ValueError(diagram)


def draw_bpt_demarcations(ax, diagram):
    if diagram == 'nii':
        x = np.linspace(-2.0, 0.04, 400)
        ax.plot(x, 0.61 / (x - 0.05) + 1.3, color='forestgreen', ls='--', lw=1.7, label='Kauffmann+03')
        x = np.linspace(-2.0, 0.46, 400)
        ax.plot(x, 0.61 / (x - 0.47) + 1.19, color='crimson', ls='-', lw=1.7, label='Kewley+01')
    elif diagram == 'sii':
        x = np.linspace(-1.6, 0.31, 400)
        ax.plot(x, 0.72 / (x - 0.32) + 1.30, color='crimson', ls='-', lw=1.7, label='Kewley+01')
        x = np.linspace(-0.31, 0.7, 200)
        ax.plot(x, 1.89 * x + 0.76, color='darkorange', ls='--', lw=1.7, label='Seyfert/LINER')


def bpt_xy(df, diagram):
    xcol = 'log_NII_Halpha_plot' if diagram == 'nii' else 'log_SII_Halpha_plot'
    ycol = 'log_OIII_Hbeta_plot'
    x = pd.to_numeric(df[xcol], errors='coerce').to_numpy(dtype=float)
    y = pd.to_numeric(df[ycol], errors='coerce').to_numpy(dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    return x, y, valid


def color_values(df, color_key):
    specs = {
        'sigma': ('sigma_mean_kms', r'$\sigma$ (km s$^{-1}$)', Normalize, {}),
        'halpha_snr': ('SNR_Halpha_sum', r'H$\alpha$ S/N', LogNorm, {'vmin_floor': 0.1}),
        'region_size': ('radius_areaeq_pc', 'Region radius (pc)', Normalize, {}),
        'sii_ha': ('sii_halpha_ratio_plot', r'[SII]/H$\alpha$', Normalize, {}),
    }
    col, label, norm_cls, opts = specs[color_key]
    vals = pd.to_numeric(df[col], errors='coerce').to_numpy(dtype=float) if col in df.columns else np.full(len(df), np.nan)
    finite = np.isfinite(vals)
    if norm_cls is LogNorm:
        positive = finite & (vals > opts.get('vmin_floor', 0))
        if positive.any():
            vmin, vmax = np.nanpercentile(vals[positive], [2, 98])
            vmin = max(vmin, opts.get('vmin_floor', 0.1))
            norm = LogNorm(vmin=vmin, vmax=vmax)
        else:
            norm = LogNorm(vmin=0.1, vmax=1.0)
    else:
        if finite.any():
            vmin, vmax = np.nanpercentile(vals[finite], [2, 98])
            norm = Normalize(vmin=vmin, vmax=vmax)
        else:
            norm = Normalize(vmin=0, vmax=1)
    return vals, label, norm


def _style_bpt_axis(ax, diagram):
    xlim, ylim, xlabel = bpt_limits(diagram)
    draw_bpt_demarcations(ax, diagram)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(r'log([OIII]5007/H$\beta$)')
    ax.minorticks_on()
    ax.tick_params(direction='in', which='both', top=True, right=True)
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)


def plot_bpt_property_pair(df, mode, color_key):
    vals, cbar_label, norm = color_values(df, color_key)
    low = df['low_snr_any_bpt_line'].to_numpy(dtype=bool)
    low_count = int(low.sum())
    fig, axes = plt.subplots(1, 2, figsize=(13.4, 5.8), sharey=True, constrained_layout=True)
    scatter = None
    for ax, diagram, label in zip(axes, ['nii', 'sii'], ['[NII] BPT', '[SII] BPT']):
        x, y, valid = bpt_xy(df, diagram)
        low_valid = valid & low
        scatter = ax.scatter(
            x[valid], y[valid], c=vals[valid], cmap='rainbow', norm=norm,
            s=18, alpha=0.72, linewidths=0, rasterized=True,
        )
        ax.scatter(
            x[low_valid], y[low_valid], facecolors='none', edgecolors='black',
            s=42, linewidths=0.8, label=f'Low S/N in any line: {low_count}', zorder=5,
        )
        _style_bpt_axis(ax, diagram)
        ax.set_title(label)
    fig.suptitle(f'{mode}: BPT diagrams colored by {cbar_label}; low S/N={low_count}', y=1.02)
    cb = fig.colorbar(scatter, ax=axes, pad=0.02, fraction=0.035)
    cb.set_label(cbar_label)
    axes[0].legend(frameon=False, fontsize=9, loc='lower left')
    out = mode_output_dir(mode, 'bpt') / f'{mode}_bpt_nii_sii_colored_by_{color_key}.png'
    fig.savefig(out, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return out


def plot_bpt_category_pair(df, mode):
    low = df['low_snr_any_bpt_line'].to_numpy(dtype=bool)
    sf = df['bpt_sf_high_snr'].to_numpy(dtype=bool)
    high_non_sf = df['high_snr_non_bpt_sf'].to_numpy(dtype=bool)
    fig, axes = plt.subplots(1, 2, figsize=(13.4, 5.8), sharey=True, constrained_layout=True)
    # User-requested switch: selected BPT SF are black, high-S/N non-SF are blue.
    layers = [
        (low, '0.72', 18, 'Low S/N in any line'),
        (high_non_sf, 'dodgerblue', 20, 'S/N>3 all lines, not BPT SF'),
        (sf, 'black', 24, 'S/N>3 all lines, BPT SF'),
    ]
    for ax, diagram, label in zip(axes, ['nii', 'sii'], ['[NII] BPT', '[SII] BPT']):
        x, y, valid = bpt_xy(df, diagram)
        for mask, color, size, legend_label in layers:
            select = valid & mask
            ax.scatter(
                x[select], y[select], s=size, c=color, alpha=0.78,
                linewidths=0, label=f'{legend_label} (N={int(mask.sum())})', rasterized=True,
            )
        _style_bpt_axis(ax, diagram)
        ax.set_title(label)
    fig.suptitle(f'{mode}: BPT SF black, high-S/N non-SF blue, low S/N grey; low S/N={int(low.sum())}', y=1.02)
    axes[0].legend(frameon=False, fontsize=9, loc='lower left')
    out = mode_output_dir(mode, 'bpt') / f'{mode}_bpt_nii_sii_snr_bptsf_categories.png'
    fig.savefig(out, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return out


def plot_bpt_nodig_sf_highlight_pair(df, mode='dig_subtracted'):
    highlight = df['nodig_sf_not_dig_sf_high_snr'].to_numpy(dtype=bool)
    low = df['low_snr_any_bpt_line'].to_numpy(dtype=bool)
    fig, axes = plt.subplots(1, 2, figsize=(13.4, 5.8), sharey=True, constrained_layout=True)
    for ax, diagram, label in zip(axes, ['nii', 'sii'], ['[NII] BPT', '[SII] BPT']):
        x, y, valid = bpt_xy(df, diagram)
        ax.scatter(x[valid], y[valid], s=18, c='0.72', alpha=0.65, linewidths=0, rasterized=True, label='All regions')
        select = valid & highlight
        ax.scatter(
            x[select], y[select], s=58, facecolors='none', edgecolors='black',
            linewidths=1.25, label=f'BPT SF before DIG, not after (N={int(highlight.sum())})', zorder=6,
        )
        low_select = valid & low
        ax.scatter(
            x[low_select], y[low_select], s=28, facecolors='none', edgecolors='0.35',
            linewidths=0.55, alpha=0.55, label=f'Low S/N any line (N={int(low.sum())})', zorder=4,
        )
        _style_bpt_axis(ax, diagram)
        ax.set_title(label)
    fig.suptitle(f'{mode}: regions that are BPT SF before DIG correction but not after', y=1.02)
    axes[0].legend(frameon=False, fontsize=9, loc='lower left')
    out = mode_output_dir(mode, 'bpt') / f'{mode}_bpt_nii_sii_nodig_sf_not_dig_sf_black_outline.png'
    fig.savefig(out, dpi=300, bbox_inches='tight')
    plt.close(fig)
    return out


## DIG Subtraction Comparison Plots

These cells compare each primary region before and after DIG subtraction using the paired no-DIG and DIG-subtracted flux columns in the DIG-subtracted catalog.

In [6]:

def comparison_output_dir(subdir=None):
    out = OUTPUT_ROOT / 'comparison'
    if subdir is not None:
        out = out / subdir
    out.mkdir(parents=True, exist_ok=True)
    return out


def _classification_column(df, mode):
    if mode == 'no_dig':
        if 'BPT_class_sum_nodig_dered' in df.columns:
            return df['BPT_class_sum_nodig_dered'].astype(str)
        return pd.Series(df['BPT_class_nodig_from_flux'], index=df.index).astype(str)
    if mode == 'dig_subtracted':
        if 'BPT_class_sum_digsub_dered' in df.columns:
            return df['BPT_class_sum_digsub_dered'].astype(str)
        return df['BPT_class_sum_dered'].astype(str)
    raise ValueError(mode)


def paired_comparison_catalog():
    df = catalogs['dig_subtracted'].copy()
    df['BPT_class_no_dig'] = _classification_column(df, 'no_dig').str.strip().replace({'nan': 'Unclassified'})
    df['BPT_class_dig_subtracted'] = _classification_column(df, 'dig_subtracted').str.strip().replace({'nan': 'Unclassified'})
    for mode, suffix in [('no_dig', 'sum_dered_nodig'), ('dig_subtracted', 'sum_dered')]:
        df[f'log_NII_Halpha_{mode}'] = np.log10(_numeric(df, f'F_[NII]6583_{suffix}') / _numeric(df, f'F_Halpha_{suffix}'))
        df[f'log_SII_Halpha_{mode}'] = np.log10(
            (_numeric(df, f'F_[SII]6716_{suffix}') + _numeric(df, f'F_[SII]6731_{suffix}')) / _numeric(df, f'F_Halpha_{suffix}')
        )
        df[f'log_OIII_Hbeta_{mode}'] = np.log10(_numeric(df, f'F_[OIII]5007_{suffix}') / _numeric(df, f'F_Hbeta_{suffix}'))
    flux_factor = _numeric(df, 'L_Ha_sum_dered') / _numeric(df, 'F_Halpha_sum_dered')
    df['L_Ha_sum_dered_no_dig'] = _numeric(df, 'F_Halpha_sum_dered_nodig') * flux_factor
    df['log_L_Ha_sum_dered_no_dig'] = np.log10(df['L_Ha_sum_dered_no_dig'])
    df['L_Ha_sum_dered_dig_subtracted'] = _numeric(df, 'L_Ha_sum_dered')
    df['log_L_Ha_sum_dered_dig_subtracted'] = _numeric(df, 'log_L_Ha_sum_dered')
    df['Z_N2_M2013_no_dig'] = marino_n2_metallicity(df, 'sum_dered_nodig')
    df['Z_O3N2_M2013_no_dig'] = marino_o3n2_metallicity(df, 'sum_dered_nodig')
    df['Z_N2_M2013_dig_subtracted'] = marino_n2_metallicity(df, 'sum_dered')
    df['Z_O3N2_M2013_dig_subtracted'] = marino_o3n2_metallicity(df, 'sum_dered')
    return df


def marino_n2_metallicity(df, flux_suffix):
    with np.errstate(divide='ignore', invalid='ignore'):
        n2 = np.log10(_numeric(df, f'F_[NII]6583_{flux_suffix}') / _numeric(df, f'F_Halpha_{flux_suffix}'))
        z = 8.743 + 0.462 * n2
    return np.where((z >= 7.6) & (z <= 8.8), z, np.nan)


def marino_o3n2_metallicity(df, flux_suffix):
    with np.errstate(divide='ignore', invalid='ignore'):
        o3n2 = np.log10(
            (_numeric(df, f'F_[OIII]5007_{flux_suffix}') / _numeric(df, f'F_Hbeta_{flux_suffix}')) /
            (_numeric(df, f'F_[NII]6583_{flux_suffix}') / _numeric(df, f'F_Halpha_{flux_suffix}'))
        )
        z = 8.533 - 0.214 * o3n2
    return np.where((z >= 7.6) & (z <= 8.8), z, np.nan)


def fit_line(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    good = np.isfinite(x) & np.isfinite(y)
    if good.sum() < 2:
        return np.nan, np.nan, good
    slope, intercept = np.polyfit(x[good], y[good], 1)
    return slope, intercept, good


paired = paired_comparison_catalog()
print(f'Paired primary regions available for before/after DIG comparison: {len(paired)}')


Paired primary regions available for before/after DIG comparison: 5902


/var/folders/90/nb83zn5j00bc53x6bcv0jcwm0000gn/T/ipykernel_7717/599145158.py:26: RuntimeWarning: invalid value encountered in log10
  df[f'log_NII_Halpha_{mode}'] = np.log10(_numeric(df, f'F_[NII]6583_{suffix}') / _numeric(df, f'F_Halpha_{suffix}'))
/var/folders/90/nb83zn5j00bc53x6bcv0jcwm0000gn/T/ipykernel_7717/599145158.py:27: RuntimeWarning: invalid value encountered in log10
  df[f'log_SII_Halpha_{mode}'] = np.log10(
/var/folders/90/nb83zn5j00bc53x6bcv0jcwm0000gn/T/ipykernel_7717/599145158.py:30: RuntimeWarning: invalid value encountered in log10
  df[f'log_OIII_Hbeta_{mode}'] = np.log10(_numeric(df, f'F_[OIII]5007_{suffix}') / _numeric(df, f'F_Hbeta_{suffix}'))
/var/folders/90/nb83zn5j00bc53x6bcv0jcwm0000gn/T/ipykernel_7717/599145158.py:26: RuntimeWarning: invalid value encountered in log10
  df[f'log_NII_Halpha_{mode}'] = np.log10(_numeric(df, f'F_[NII]6583_{suffix}') / _numeric(df, f'F_Halpha_{suffix}'))
/var/folders/90/nb83zn5j00bc53x6bcv0jcwm0000gn/T/ipykernel_7717/599145158.p

In [7]:

def classification_transition_table(df):
    valid = df['BPT_class_no_dig'].ne('Unclassified') & df['BPT_class_dig_subtracted'].ne('Unclassified')
    subset = df.loc[valid].copy()
    changed = subset['BPT_class_no_dig'].ne(subset['BPT_class_dig_subtracted'])
    total = len(subset)
    changed_n = int(changed.sum())
    print(f'BPT classifications compared for {total} regions with valid before/after classes.')
    print(f'Changed classification after DIG subtraction: {changed_n}/{total} ({100 * changed_n / total:.1f}%)')
    rows = []
    classes = ['Star-forming', 'Composite', 'AGN/Shock']
    for before in classes:
        before_total = int((subset['BPT_class_no_dig'] == before).sum())
        for after in classes:
            n = int(((subset['BPT_class_no_dig'] == before) & (subset['BPT_class_dig_subtracted'] == after)).sum())
            if before == after or n == 0:
                continue
            rows.append({
                'No DIG class': before,
                'DIG-subtracted class': after,
                'N': n,
                '% of all valid': 100 * n / total if total else np.nan,
                '% of no-DIG source class': 100 * n / before_total if before_total else np.nan,
            })
    transition_df = pd.DataFrame(rows).sort_values(['No DIG class', 'DIG-subtracted class']).reset_index(drop=True)
    print('\nClassification switches:')
    if transition_df.empty:
        print('No regions switch classification.')
    else:
        print(transition_df.to_string(index=False, formatters={
            '% of all valid': '{:.1f}'.format,
            '% of no-DIG source class': '{:.1f}'.format,
        }))
    matrix = pd.crosstab(subset['BPT_class_no_dig'], subset['BPT_class_dig_subtracted']).reindex(index=classes, columns=classes, fill_value=0)
    return transition_df, matrix


classification_switches, classification_matrix = classification_transition_table(paired)
classification_switches.to_csv(comparison_output_dir('tables') / 'bpt_classification_switches.csv', index=False)
classification_matrix.to_csv(comparison_output_dir('tables') / 'bpt_classification_transition_matrix.csv')


BPT classifications compared for 5670 regions with valid before/after classes.
Changed classification after DIG subtraction: 104/5670 (1.8%)

Classification switches:
No DIG class DIG-subtracted class  N % of all valid % of no-DIG source class
   AGN/Shock         Star-forming  2            0.0                      1.1
   Composite            AGN/Shock 48            0.8                     15.4
   Composite         Star-forming  9            0.2                      2.9
Star-forming            AGN/Shock 12            0.2                      0.2
Star-forming            Composite 33            0.6                      0.6


In [8]:

COMPARISON_COLORS = {
    'no_dig': '#1f77b4',
    'dig_subtracted': '#d62728',
}
COMPARISON_LABELS = {
    'no_dig': 'No DIG subtraction',
    'dig_subtracted': 'DIG subtracted',
}


def bpt_shift_arrays(df, diagram):
    if diagram == 'nii':
        xbase = 'log_NII_Halpha'
    elif diagram == 'sii':
        xbase = 'log_SII_Halpha'
    else:
        raise ValueError(diagram)
    x0 = pd.to_numeric(df[f'{xbase}_no_dig'], errors='coerce').to_numpy(dtype=float)
    y0 = pd.to_numeric(df['log_OIII_Hbeta_no_dig'], errors='coerce').to_numpy(dtype=float)
    x1 = pd.to_numeric(df[f'{xbase}_dig_subtracted'], errors='coerce').to_numpy(dtype=float)
    y1 = pd.to_numeric(df['log_OIII_Hbeta_dig_subtracted'], errors='coerce').to_numpy(dtype=float)
    good = np.isfinite(x0) & np.isfinite(y0) & np.isfinite(x1) & np.isfinite(y1)
    return x0, y0, x1, y1, good


def add_bpt_shift_panel(ax, df, diagram):
    x0, y0, x1, y1, good = bpt_shift_arrays(df, diagram)
    _style_bpt_axis(ax, diagram)
    ax.quiver(
        x0[good], y0[good], x1[good] - x0[good], y1[good] - y0[good],
        angles='xy', scale_units='xy', scale=1, width=0.0015,
        headwidth=0.1, headlength=0.1, headaxislength=1.0,
        color='k', alpha=0.20, zorder=1,
    )
    ax.scatter(
        x0[good], y0[good], s=12, c=COMPARISON_COLORS['no_dig'], alpha=0.45,
        linewidths=0, rasterized=True, label=COMPARISON_LABELS['no_dig'], zorder=2,
    )
    ax.scatter(
        x1[good], y1[good], s=12, c=COMPARISON_COLORS['dig_subtracted'], alpha=0.45,
        linewidths=0, rasterized=True, label=COMPARISON_LABELS['dig_subtracted'], zorder=3,
    )
    mean_dx = np.nanmean(x1[good] - x0[good])
    mean_dy = np.nanmean(y1[good] - y0[good])
    mean_shift_length = np.nanmean(np.hypot(x1[good] - x0[good], y1[good] - y0[good]))
    mean_x1, mean_y1 = np.nanmean(x1[good]), np.nanmean(y1[good])
    arrow_tail = (mean_x1 - mean_dx, mean_y1 - mean_dy)
    ax.annotate(
        '', xy=(mean_x1, mean_y1), xytext=arrow_tail,
        arrowprops=dict(arrowstyle='-|>', color='black', lw=1.2, mutation_scale=10),
        zorder=8,
    )
    ax.scatter([arrow_tail[0]], [arrow_tail[1]], s=95, c=COMPARISON_COLORS['no_dig'], edgecolors='black', linewidths=1.0, zorder=9)
    ax.scatter([mean_x1], [mean_y1], s=95, c=COMPARISON_COLORS['dig_subtracted'], edgecolors='black', linewidths=1.0, zorder=9)
    ax.text(
        0.03, 0.97,
        rf'Mean shift: $\Delta x$={mean_dx:.3f}, $\Delta y$={mean_dy:.3f}' + '\n' + f'Mean pair length={mean_shift_length:.3f}',
        transform=ax.transAxes, ha='left', va='top', fontsize=9,
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=3),
    )
    label = '[NII]' if diagram == 'nii' else '[SII]'
    ax.set_title(f'{label} BPT shift after DIG subtraction (N={int(good.sum())})')
    return {
        'diagram': diagram,
        'N': int(good.sum()),
        'mean_dx': mean_dx,
        'mean_dy': mean_dy,
        'mean_pair_length': mean_shift_length,
        'mean_dig_subtracted_x': mean_x1,
        'mean_dig_subtracted_y': mean_y1,
    }


def plot_bpt_shift_arrows(df):
    out_dir = comparison_output_dir('bpt')
    shift_rows = []

    fig, axes = plt.subplots(1, 2, figsize=(13.8, 6.2), sharey=True, constrained_layout=True)
    for ax, diagram in zip(axes, ['nii', 'sii']):
        shift_rows.append(add_bpt_shift_panel(ax, df, diagram))
    axes[0].legend(frameon=False, loc='lower left')
    combined = out_dir / 'BPT_no_dig_to_dig_subtracted_shift_arrows_NII_SII.png'
    fig.savefig(combined, dpi=300, bbox_inches='tight')
    plt.close(fig)

    separate_paths = []
    for diagram in ['nii', 'sii']:
        fig, ax = plt.subplots(figsize=(7.4, 6.4), constrained_layout=True)
        add_bpt_shift_panel(ax, df, diagram)
        ax.legend(frameon=False, loc='lower left')
        label = 'NII' if diagram == 'nii' else 'SII'
        out = out_dir / f'BPT_no_dig_to_dig_subtracted_shift_arrows_{label}.png'
        fig.savefig(out, dpi=300, bbox_inches='tight')
        plt.close(fig)
        separate_paths.append(out)

    shifts = pd.DataFrame(shift_rows)
    shifts.to_csv(comparison_output_dir('tables') / 'bpt_mean_shift_vectors.csv', index=False)
    return [combined, *separate_paths], shifts


bpt_shift_paths, bpt_shift_vectors = plot_bpt_shift_arrows(paired)
print('Wrote BPT shift plots:')
for path in bpt_shift_paths:
    print(f'  {path}')
print(bpt_shift_vectors.to_string(index=False, formatters={
    'mean_dx': '{:.4f}'.format,
    'mean_dy': '{:.4f}'.format,
    'mean_pair_length': '{:.4f}'.format,
    'mean_dig_subtracted_x': '{:.4f}'.format,
    'mean_dig_subtracted_y': '{:.4f}'.format,
}))


Wrote BPT shift plots:
  bpt_comparison/comparison/bpt/BPT_no_dig_to_dig_subtracted_shift_arrows_NII_SII.png
  bpt_comparison/comparison/bpt/BPT_no_dig_to_dig_subtracted_shift_arrows_NII.png
  bpt_comparison/comparison/bpt/BPT_no_dig_to_dig_subtracted_shift_arrows_SII.png
diagram    N mean_dx mean_dy mean_pair_length mean_dig_subtracted_x mean_dig_subtracted_y
    nii 5670 -0.0212 -0.0028           0.0407               -0.7278               -0.1767
    sii 5636 -0.0268 -0.0029           0.0445               -0.4502               -0.1777


In [9]:

def luminosity_function_pdf_points(logL, bins):
    logL = np.asarray(logL, dtype=float)
    logL = logL[np.isfinite(logL)]
    if len(logL) == 0:
        return np.array([]), np.array([]), np.array([])
    counts, edges = np.histogram(logL, bins=bins)
    centers = 0.5 * (edges[:-1] + edges[1:])
    widths = np.diff(edges)
    pdf = counts / (np.sum(counts) * widths)
    detected = counts > 0
    return centers[detected], pdf[detected], counts[detected]


def empirical_cdf(sorted_x):
    n = len(sorted_x)
    return np.arange(1, n + 1) / n


def model_cdf_continuous_powerlaw(x, xmin, alpha):
    return 1.0 - (x / xmin) ** (1.0 - alpha)


def ks_statistic_continuous_powerlaw(sorted_x, xmin, alpha):
    return np.max(np.abs(empirical_cdf(sorted_x) - model_cdf_continuous_powerlaw(sorted_x, xmin, alpha)))


def mle_alpha_continuous_powerlaw(x_tail, xmin):
    n = len(x_tail)
    s = np.sum(np.log(x_tail / xmin))
    if s <= 0:
        return np.nan
    return 1.0 + n / s


def gaussian(x, A, mu, sigma):
    return A * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


def santoro_powerlaw_fit_fast(
    L,
    alpha_range=(1.0, 3.0),
    xmin_stride=1,
    min_tail_n=30,
    n_bootstrap=1000,
    random_seed=42,
    verbose=False,
):
    rng = np.random.default_rng(random_seed)
    L = np.asarray(L, dtype=float)
    L = np.sort(L[np.isfinite(L) & (L > 0)])
    if len(L) < min_tail_n:
        raise ValueError('Not enough positive luminosities.')

    alpha_min, alpha_max = alpha_range
    med = np.median(L)
    sig = np.std(L, ddof=1)
    xmin_lo = med - sig
    xmin_hi = med + sig
    xmin_candidates = np.unique(L[(L >= xmin_lo) & (L <= xmin_hi)])
    if len(xmin_candidates) == 0:
        raise ValueError('No xmin candidates found in median +/- 1 sigma range.')
    xmin_candidates = xmin_candidates[::xmin_stride]

    best = None
    fit_rows = []
    for xmin in xmin_candidates:
        i0 = np.searchsorted(L, xmin, side='left')
        x_tail = L[i0:]
        n_tail = len(x_tail)
        if n_tail < min_tail_n:
            continue
        alpha = mle_alpha_continuous_powerlaw(x_tail, xmin)
        if not np.isfinite(alpha) or not (alpha_min <= alpha <= alpha_max):
            continue
        ks = ks_statistic_continuous_powerlaw(x_tail, xmin, alpha)
        row = {'xmin': xmin, 'alpha': alpha, 'ks': ks, 'n_tail': n_tail}
        fit_rows.append(row)
        if best is None or ks < best['ks']:
            best = row

    if best is None:
        raise ValueError('No valid fit found. Try decreasing xmin_stride or min_tail_n.')

    bootstrap_xmins = []
    bootstrap_alphas = []
    for _ in range(n_bootstrap):
        Lb = rng.choice(L, size=len(L), replace=True)
        Lb.sort()
        med_b = np.median(Lb)
        sig_b = np.std(Lb, ddof=1)
        xmin_candidates_b = np.unique(Lb[(Lb >= med_b - sig_b) & (Lb <= med_b + sig_b)])
        xmin_candidates_b = xmin_candidates_b[::xmin_stride]
        best_b = None
        for xmin_b in xmin_candidates_b:
            i0_b = np.searchsorted(Lb, xmin_b, side='left')
            x_tail_b = Lb[i0_b:]
            n_tail_b = len(x_tail_b)
            if n_tail_b < min_tail_n:
                continue
            alpha_b = mle_alpha_continuous_powerlaw(x_tail_b, xmin_b)
            if not np.isfinite(alpha_b) or not (alpha_min <= alpha_b <= alpha_max):
                continue
            ks_b = ks_statistic_continuous_powerlaw(x_tail_b, xmin_b, alpha_b)
            if best_b is None or ks_b < best_b['ks']:
                best_b = {'xmin': xmin_b, 'alpha': alpha_b, 'ks': ks_b}
        if best_b is not None:
            bootstrap_xmins.append(best_b['xmin'])
            bootstrap_alphas.append(best_b['alpha'])

    bootstrap_xmins = np.asarray(bootstrap_xmins, dtype=float)
    bootstrap_alphas = np.asarray(bootstrap_alphas, dtype=float)
    xmin_err = np.std(bootstrap_xmins, ddof=1) if len(bootstrap_xmins) > 1 else np.nan

    if verbose:
        print('Fast Santoro/Clauset fit')
        print(f'N total           = {len(L)}')
        print(f"Best xmin         = {best['xmin']:.3e}")
        print(f"Best alpha        = {best['alpha']:.4f}")
        print(f"Best KS           = {best['ks']:.4f}")
        print(f"N above xmin      = {best['n_tail']}")
        print(f'xmin uncertainty  = {xmin_err:.3e}')
        print(f'Candidates tested = {len(fit_rows)}')

    return {
        'L': L,
        'xmin_best': best['xmin'],
        'alpha_best': best['alpha'],
        'ks_best': best['ks'],
        'n_tail_best': best['n_tail'],
        'xmin_err': xmin_err,
        'bootstrap_xmins': bootstrap_xmins,
        'bootstrap_alphas': bootstrap_alphas,
        'fit_rows': fit_rows,
        'xmin_search_range': (xmin_lo, xmin_hi),
    }


def alpha_uncertainty_from_likelihood(L, xmin, alpha_best, alpha_range=(1.0, 3.0), n_alpha_grid=400):
    x_tail = np.sort(np.asarray(L, dtype=float)[np.asarray(L, dtype=float) >= xmin])
    alpha_grid = np.linspace(alpha_range[0], alpha_range[1], n_alpha_grid)
    n = len(x_tail)
    sum_logx = np.sum(np.log(x_tail))
    log_xmin = np.log(xmin)
    valid = alpha_grid > 1
    loglike = np.full_like(alpha_grid, -np.inf, dtype=float)
    a = alpha_grid[valid]
    loglike[valid] = n * np.log(a - 1.0) + n * (a - 1.0) * log_xmin - a * sum_logx
    rel_like = np.exp(loglike - np.nanmax(loglike))

    peak_mask = rel_like > 0.05
    x_fit = alpha_grid[peak_mask]
    y_fit = rel_like[peak_mask]
    if len(x_fit) < 5:
        x_fit = alpha_grid[valid]
        y_fit = rel_like[valid]

    try:
        popt, _ = curve_fit(
            gaussian, x_fit, y_fit,
            p0=[1.0, alpha_best, 0.1],
            bounds=([0.0, alpha_range[0], 1e-4], [np.inf, alpha_range[1], 10.0]),
            maxfev=10000,
        )
        sigma_alpha = popt[2]
    except Exception:
        w = rel_like[valid] / np.sum(rel_like[valid])
        mu = np.sum(w * alpha_grid[valid])
        sigma_alpha = np.sqrt(np.sum(w * (alpha_grid[valid] - mu) ** 2))
        popt = None
    return sigma_alpha, alpha_grid, rel_like, popt


def _powerlaw_pdf_per_dex(logL_grid, fit_results):
    alpha = fit_results['alpha_best']
    xmin = fit_results['xmin_best']
    n_total = len(fit_results['L'])
    n_tail = fit_results['n_tail_best']
    L_grid = 10.0 ** np.asarray(logL_grid, dtype=float)
    pdf = np.full_like(L_grid, np.nan, dtype=float)
    tail = L_grid >= xmin
    tail_fraction = n_tail / n_total
    pdf[tail] = (
        tail_fraction * np.log(10.0) * (alpha - 1.0) *
        (xmin ** (alpha - 1.0)) * (L_grid[tail] ** (1.0 - alpha))
    )
    return pdf


def _positive_luminosities(values):
    values = np.asarray(values, dtype=float)
    return values[np.isfinite(values) & (values > 0)]


def lf_mode_mask(df, mode):
    class_col = 'BPT_class_no_dig' if mode == 'no_dig' else 'BPT_class_dig_subtracted'
    snr_suffix = 'sum_nodig' if mode == 'no_dig' else 'sum'
    snr_cols = [
        f'SNR_Halpha_{snr_suffix}', f'SNR_Hbeta_{snr_suffix}', f'SNR_[OIII]5007_{snr_suffix}',
        f'SNR_[NII]6583_{snr_suffix}', f'SNR_[SII]6716_{snr_suffix}', f'SNR_[SII]6731_{snr_suffix}',
    ]
    high_snr = np.ones(len(df), dtype=bool)
    for col in snr_cols:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors='coerce').to_numpy(dtype=float)
            high_snr &= np.isfinite(vals) & (vals >= SNR_CUT)
    return high_snr & df[class_col].eq('Star-forming').to_numpy(dtype=bool)


LF_XMIN_STRIDE = 10
LF_N_BOOTSTRAP = 10


def fit_lf_like_notebook8(L, label, random_seed=42):
    results = santoro_powerlaw_fit_fast(
        L,
        alpha_range=(1.0, 3.0),
        xmin_stride=LF_XMIN_STRIDE,
        min_tail_n=30,
        n_bootstrap=LF_N_BOOTSTRAP,
        random_seed=random_seed,
        verbose=False,
    )
    sigma_alpha, alpha_grid, rel_like, gfit = alpha_uncertainty_from_likelihood(
        results['L'],
        results['xmin_best'],
        results['alpha_best'],
        alpha_range=(1.0, 3.0),
        n_alpha_grid=400,
    )
    results['alpha_err'] = sigma_alpha
    results['label'] = label
    return results


def plot_luminosity_functions(df):
    logL_bins = np.arange(33.0, 40.6, 0.2)
    specs = [
        dict(mode='no_dig', label=COMPARISON_LABELS['no_dig'], lum_col='L_Ha_sum_dered_no_dig', marker='o'),
        dict(mode='dig_subtracted', label=COMPARISON_LABELS['dig_subtracted'], lum_col='L_Ha_sum_dered_dig_subtracted', marker='s'),
    ]
    fig, ax = plt.subplots(figsize=(8.8, 6.5), constrained_layout=True)
    fit_rows = []

    for index, spec in enumerate(specs):
        mask = lf_mode_mask(df, spec['mode'])
        L_values = _positive_luminosities(pd.to_numeric(df.loc[mask, spec['lum_col']], errors='coerce').to_numpy(dtype=float))
        logL_values = np.log10(L_values)
        centers, pdf, counts = luminosity_function_pdf_points(logL_values, logL_bins)
        if len(centers) == 0:
            print(f"Skipping {spec['label']}: no finite luminosities")
            continue

        fit_results = fit_lf_like_notebook8(L_values, spec['label'], random_seed=42 + index)
        alpha = fit_results['alpha_best']
        alpha_err = fit_results['alpha_err']
        slope = -alpha
        log_lmin = np.log10(fit_results['xmin_best'])
        log_lmin_err = fit_results['xmin_err'] / (fit_results['xmin_best'] * np.log(10.0))
        color = COMPARISON_COLORS[spec['mode']]
        label = (
            f"{spec['label']} (SF, S/N>{SNR_CUT:g}; N={len(L_values)})\n"
            rf"$\alpha={alpha:.2f}\pm{alpha_err:.2f}$, slope$={slope:.2f}$, "
            rf"$\log L_{{\min}}={log_lmin:.2f}\pm{log_lmin_err:.2f}$"
        )
        ax.plot(
            centers, pdf,
            marker=spec['marker'], markersize=5.8, linewidth=2.1,
            color=color, label=label,
        )
        fit_logL = np.linspace(log_lmin, max(np.nanmax(centers), log_lmin + 0.2), 200)
        ax.plot(fit_logL, _powerlaw_pdf_per_dex(fit_logL, fit_results), color=color, linestyle='--', linewidth=1.8, alpha=0.9)
        ax.axvline(log_lmin, color=color, linestyle=':', linewidth=1.8, alpha=0.9)
        fit_rows.append({
            'Mode': spec['label'],
            'Sample': f'Star-forming, S/N>{SNR_CUT:g}',
            'N': len(L_values),
            'N tail': fit_results['n_tail_best'],
            'Alpha': alpha,
            'Alpha error': alpha_err,
            'Slope': slope,
            'Slope error': alpha_err,
            'Lmin': fit_results['xmin_best'],
            'log Lmin': log_lmin,
            'log Lmin error': log_lmin_err,
            'KS': fit_results['ks_best'],
            'xmin stride': LF_XMIN_STRIDE,
            'N bootstrap': LF_N_BOOTSTRAP,
        })

    ax.set_xlabel(r'log$_{10}$[$L_{\rm H\alpha}$ (erg s$^{-1}$)]')
    ax.set_ylabel('Empirical PDF per dex')
    ax.set_yscale('log')
    ax.minorticks_on()
    ax.legend(frameon=False, fontsize=9.0, loc='best')
    ax.set_title(r'H$\alpha$ luminosity function before/after DIG subtraction')
    out = comparison_output_dir('luminosity_function') / 'Halpha_luminosity_function_no_dig_vs_dig_subtracted.png'
    fig.savefig(out, dpi=300, bbox_inches='tight')
    plt.close(fig)
    fit_df = pd.DataFrame(fit_rows)
    fit_df.to_csv(comparison_output_dir('tables') / 'luminosity_function_slopes.csv', index=False)
    return out, fit_df


lf_path, lf_slopes = plot_luminosity_functions(paired)
print(f'Wrote luminosity function plot: {lf_path}')
print(lf_slopes.to_string(index=False, formatters={
    'Alpha': '{:.3f}'.format,
    'Alpha error': '{:.3f}'.format,
    'Slope': '{:.3f}'.format,
    'Slope error': '{:.3f}'.format,
    'log Lmin': '{:.3f}'.format,
    'log Lmin error': '{:.3f}'.format,
    'KS': '{:.3f}'.format,
}))


Wrote luminosity function plot: bpt_comparison/comparison/luminosity_function/Halpha_luminosity_function_no_dig_vs_dig_subtracted.png
              Mode  Slope Intercept Fit min log L    N
No DIG subtraction  0.106    -3.058          38.6 5807
    DIG subtracted -0.079     4.155          38.6 5807


In [10]:

def mode_sf_mask_for_metallicity(df, mode):
    class_col = 'BPT_class_no_dig' if mode == 'no_dig' else 'BPT_class_dig_subtracted'
    snr_suffix = 'sum_nodig' if mode == 'no_dig' else 'sum'
    snr_cols = [
        f'SNR_Halpha_{snr_suffix}', f'SNR_Hbeta_{snr_suffix}', f'SNR_[OIII]5007_{snr_suffix}',
        f'SNR_[NII]6583_{snr_suffix}', f'SNR_[SII]6716_{snr_suffix}', f'SNR_[SII]6731_{snr_suffix}',
    ]
    high_snr = np.ones(len(df), dtype=bool)
    for col in snr_cols:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors='coerce').to_numpy(dtype=float)
            high_snr &= np.isfinite(vals) & (vals >= SNR_CUT)
    return high_snr & df[class_col].eq('Star-forming').to_numpy(dtype=bool)


def plot_marino_metallicity_comparison(df):
    panels = [
        ('Z_N2_M2013', 'Marino+13 N2'),
        ('Z_O3N2_M2013', 'Marino+13 O3N2'),
    ]
    fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.4), sharex=True, sharey=True, constrained_layout=True)
    slope_rows = []
    r = pd.to_numeric(df['R_gal_kpc'], errors='coerce').to_numpy(dtype=float)
    for ax, (base, title) in zip(axes, panels):
        for mode in ['no_dig', 'dig_subtracted']:
            z = pd.to_numeric(df[f'{base}_{mode}'], errors='coerce').to_numpy(dtype=float)
            mask = mode_sf_mask_for_metallicity(df, mode) & np.isfinite(r) & np.isfinite(z)
            slope, intercept, good = fit_line(r[mask], z[mask])
            ax.scatter(r[mask], z[mask], s=14, color=COMPARISON_COLORS[mode], alpha=0.42, linewidths=0, rasterized=True, label=f'{COMPARISON_LABELS[mode]} (N={int(mask.sum())})')
            if np.isfinite(slope):
                xfit = np.linspace(np.nanmin(r[mask]), np.nanmax(r[mask]), 100)
                ax.plot(xfit, intercept + slope * xfit, color=COMPARISON_COLORS[mode], lw=2.2, label=f'{COMPARISON_LABELS[mode]} slope={slope:.3f}')
            slope_rows.append({'Calibration': title, 'Mode': COMPARISON_LABELS[mode], 'Slope dex/kpc': slope, 'Intercept': intercept, 'N': int(mask.sum())})
        ax.set_title(title)
        ax.set_xlabel(r'R$_{gal}$ (kpc)')
        ax.minorticks_on()
        ax.tick_params(direction='in', which='both', top=True, right=True)
        ax.legend(frameon=False, fontsize=8, loc='best')
    axes[0].set_ylabel(r'12 + log(O/H)')
    fig.suptitle('Marino et al. (2013) metallicity gradients before/after DIG subtraction', y=1.02)
    out = comparison_output_dir('metallicity') / 'Marino2013_metallicity_gradients_no_dig_vs_dig_subtracted.png'
    fig.savefig(out, dpi=300, bbox_inches='tight')
    plt.close(fig)
    slopes = pd.DataFrame(slope_rows)
    slopes.to_csv(comparison_output_dir('tables') / 'marino2013_metallicity_gradient_slopes.csv', index=False)
    return out, slopes


metallicity_path, marino_slopes = plot_marino_metallicity_comparison(paired)
print(f'Wrote Marino+13 metallicity comparison plot: {metallicity_path}')
print(marino_slopes.to_string(index=False, formatters={'Slope dex/kpc': '{:.4f}'.format, 'Intercept': '{:.3f}'.format}))


Wrote Marino+13 metallicity comparison plot: bpt_comparison/comparison/metallicity/Marino2013_metallicity_gradients_no_dig_vs_dig_subtracted.png
   Calibration               Mode Slope dex/kpc Intercept    N
  Marino+13 N2 No DIG subtraction       -0.0324     8.515 4578
  Marino+13 N2     DIG subtracted       -0.0338     8.511 4546
Marino+13 O3N2 No DIG subtraction       -0.0313     8.520 4580
Marino+13 O3N2     DIG subtracted       -0.0322     8.522 4549


In [11]:
def field_image_path(field):
    field_dir = FIELD_DIR[field]
    if field in HALPHA_ONLY_FIELDS:
        return field_dir / HA_PATTERN.format(FIELD=field)
    return field_dir / HAOIII_PATTERN.format(FIELD=field)


def load_field_image_and_wcs(field):
    path = field_image_path(field)
    data = np.squeeze(fits.getdata(path)).astype(float)
    header = fits.getheader(path)
    data[~np.isfinite(data)] = np.nan
    return data, WCS(header), path


def build_haiii_mosaic(fields=FIELDS):
    inputs = []
    for field in fields:
        data, wcs, path = load_field_image_and_wcs(field)
        inputs.append((data, wcs))
    mosaic_wcs, shape_out = find_optimal_celestial_wcs(inputs, auto_rotate=True)
    mosaic_sum = np.zeros(shape_out, dtype=float)
    footprint_sum = np.zeros(shape_out, dtype=float)
    for data, wcs in inputs:
        reproj, footprint = reproject_interp((data, wcs), mosaic_wcs, shape_out=shape_out)
        good = np.isfinite(reproj) & (footprint > 0)
        mosaic_sum[good] += reproj[good]
        footprint_sum[good] += footprint[good]
    mosaic = np.full(shape_out, np.nan, dtype=float)
    good = footprint_sum > 0
    mosaic[good] = mosaic_sum[good] / footprint_sum[good]
    return mosaic, mosaic_wcs


MOSAIC_CACHE_DIR = OUTPUT_ROOT / 'cache'
MOSAIC_CACHE_DIR.mkdir(parents=True, exist_ok=True)
MOSAIC_CACHE_DATA = MOSAIC_CACHE_DIR / 'HaOIII_background_mosaic.npy'
MOSAIC_CACHE_HEADER = MOSAIC_CACHE_DIR / 'HaOIII_background_mosaic_wcs.fits'

if MOSAIC_CACHE_DATA.exists() and MOSAIC_CACHE_HEADER.exists():
    mosaic_data = np.load(MOSAIC_CACHE_DATA)
    mosaic_wcs = WCS(fits.getheader(MOSAIC_CACHE_HEADER))
    print(f'Loaded cached background mosaic with shape {mosaic_data.shape}')
else:
    mosaic_data, mosaic_wcs = build_haiii_mosaic()
    np.save(MOSAIC_CACHE_DATA, mosaic_data)
    fits.PrimaryHDU(data=np.zeros((1, 1), dtype=float), header=mosaic_wcs.to_header()).writeto(MOSAIC_CACHE_HEADER, overwrite=True)
    print(f'Built and cached background mosaic with shape {mosaic_data.shape}')

def safe_log10_image(data):
    out = np.full_like(data, np.nan, dtype=float)
    good = np.isfinite(data) & (data > 0)
    out[good] = np.log10(data[good])
    return out

mosaic_log = safe_log10_image(mosaic_data)
mosaic_valid_mask = np.isfinite(mosaic_log)
finite_log = mosaic_log[mosaic_valid_mask]
vmin = np.nanpercentile(finite_log, 1) if finite_log.size else None
vmax = np.nanpercentile(finite_log, 99.7) if finite_log.size else None
print(f'Mosaic display log10 limits: vmin={vmin:.3f}, vmax={vmax:.3f}')


Loaded cached background mosaic with shape (8797, 5961)
Mosaic display log10 limits: vmin=-19.221, vmax=-15.386


In [12]:
def field_catalog_subset(df, field):
    return df.loc[df['field'].astype(str).str.upper().eq(str(field).upper())].copy()


def category_region_ids(field_cat, mode):
    if mode == 'snr_only':
        return {
            'low_snr': field_cat.loc[field_cat['low_snr_any_bpt_line'], 'region_id'].dropna().astype(int).to_numpy(),
            'high_snr': field_cat.loc[field_cat['high_snr_all_bpt_lines'], 'region_id'].dropna().astype(int).to_numpy(),
        }
    if mode == 'snr_bpt_sf':
        return {
            'low_snr': field_cat.loc[field_cat['low_snr_any_bpt_line'], 'region_id'].dropna().astype(int).to_numpy(),
            'high_snr_non_sf': field_cat.loc[field_cat['high_snr_non_bpt_sf'], 'region_id'].dropna().astype(int).to_numpy(),
            'bpt_sf_high_snr': field_cat.loc[field_cat['bpt_sf_high_snr'], 'region_id'].dropna().astype(int).to_numpy(),
        }
    if mode == 'nodig_sf_not_dig_sf':
        return {
            'all_regions': field_cat['region_id'].dropna().astype(int).to_numpy(),
            'nodig_sf_not_dig_sf': field_cat.loc[field_cat['nodig_sf_not_dig_sf_high_snr'], 'region_id'].dropna().astype(int).to_numpy(),
        }
    raise ValueError(mode)


BOUNDARY_STYLES = {
    'snr_only': {
        'low_snr': dict(color='0.68', lw=0.35, alpha=0.75, label='Low S/N in any BPT line'),
        'high_snr': dict(color='black', lw=0.55, alpha=0.95, label='S/N>3 in all BPT lines'),
    },
    'snr_bpt_sf': {
        'low_snr': dict(color='0.70', lw=0.35, alpha=0.75, label='Low S/N in any BPT line'),
        'high_snr_non_sf': dict(color='dodgerblue', lw=0.65, alpha=0.95, label='S/N>3 all lines, not BPT SF'),
        'bpt_sf_high_snr': dict(color='black', lw=0.80, alpha=1.0, label='S/N>3 all lines, BPT SF'),
    },
    'nodig_sf_not_dig_sf': {
        'all_regions': dict(color='0.70', lw=0.32, alpha=0.65, label='All regions'),
        'nodig_sf_not_dig_sf': dict(color='black', lw=1.15, alpha=1.0, label='BPT SF before DIG, not after'),
    },
}


def plot_transformed_contours(ax, mask, field_wcs, mosaic_wcs, style):
    if not np.any(mask):
        return
    contours = find_contours(mask.astype(float), 0.5)
    for contour in contours:
        if len(contour) < 4:
            continue
        y = contour[:, 0]
        x = contour[:, 1]
        world = field_wcs.pixel_to_world(x, y)
        xm, ym = mosaic_wcs.world_to_pixel(world)
        good = np.isfinite(xm) & np.isfinite(ym)
        if good.sum() < 4:
            continue
        ax.plot(xm[good], ym[good], color=style['color'], lw=style['lw'], alpha=style['alpha'], zorder=5)



def _parse_wr_coordinates(wr_catalog):
    coords = []
    for name in wr_catalog.get('Name (star)', []):
        text = str(name).strip()
        if not text.startswith('J') or len(text) < 17:
            continue
        compact = text[1:]
        try:
            ra = f'{compact[0:2]}:{compact[2:4]}:{compact[4:9]}'
            dec = f'{compact[9:12]}:{compact[12:14]}:{compact[14:18]}'
            coords.append(SkyCoord(ra, dec, unit=(u.hourangle, u.deg)))
        except Exception:
            continue
    if not coords:
        return None
    return SkyCoord(coords)


def _snr_coordinates(snr_catalog):
    if snr_catalog is None or len(snr_catalog) == 0:
        return None
    if {'RA', 'Dec'}.issubset(snr_catalog.columns):
        try:
            return SkyCoord(snr_catalog['RA'].astype(str), snr_catalog['Dec'].astype(str), unit=(u.hourangle, u.deg))
        except Exception:
            return None
    return None


def _pn_coordinates(pn_catalog):
    if pn_catalog is None or len(pn_catalog) == 0:
        return None
    if {'RA_deg', 'Dec_deg'}.issubset(pn_catalog.columns):
        return SkyCoord(
            pn_catalog['RA_deg'].to_numpy(dtype=float) * u.deg,
            pn_catalog['Dec_deg'].to_numpy(dtype=float) * u.deg,
        )
    if {'RA', 'Dec'}.issubset(pn_catalog.columns):
        return SkyCoord(pn_catalog['RA'].astype(str), pn_catalog['Dec'].astype(str), unit=(u.hourangle, u.deg))
    return None


def _plot_catalog_points_on_mosaic(ax, coords, marker, color, label, size, zorder=20):
    if coords is None or len(coords) == 0:
        return None
    x, y = mosaic_wcs.world_to_pixel(coords)
    xi = np.rint(x).astype(int)
    yi = np.rint(y).astype(int)
    in_frame = np.isfinite(x) & np.isfinite(y) & (xi >= 0) & (xi < mosaic_log.shape[1]) & (yi >= 0) & (yi < mosaic_log.shape[0])
    valid = in_frame & mosaic_valid_mask[yi.clip(0, mosaic_log.shape[0] - 1), xi.clip(0, mosaic_log.shape[1] - 1)]
    if not np.any(valid):
        return None
    return ax.scatter(
        x[valid], y[valid], marker=marker, s=size, c=color,
        edgecolors='black', linewidths=0.45, label=label, zorder=zorder,
    )

def plot_boundary_mosaic(df, mode, category_mode):
    fig, ax = plt.subplots(figsize=(12, 12), subplot_kw={'projection': mosaic_wcs})
    ax.imshow(mosaic_log, origin='lower', cmap='rainbow', vmin=vmin, vmax=vmax, aspect='equal')

    styles = BOUNDARY_STYLES[category_mode]
    counts = {key: int(df[key].sum()) if key in df.columns and df[key].dtype == bool else 0 for key in styles}

    for field in FIELDS:
        _, field_wcs, _ = load_field_image_and_wcs(field)
        boundary_path = BOUNDARY_DIR / f'Boundary_map_{field}.fits'
        label_map = np.squeeze(fits.getdata(boundary_path)).astype(int)
        field_cat = field_catalog_subset(df, field)
        ids_by_category = category_region_ids(field_cat, category_mode)
        for category, region_ids in ids_by_category.items():
            if len(region_ids) == 0:
                continue
            mask = np.isin(label_map, region_ids)
            plot_transformed_contours(ax, mask, field_wcs, mosaic_wcs, styles[category])

    compact_handles = []
    for handle in [
        _plot_catalog_points_on_mosaic(ax, _parse_wr_coordinates(wr_catalog), marker='*', color='darkviolet', label='WR star', size=70, zorder=30),
        _plot_catalog_points_on_mosaic(ax, _snr_coordinates(snr_catalog), marker='o', color='yellow', label='SNR', size=22, zorder=28),
        _plot_catalog_points_on_mosaic(ax, _pn_coordinates(pn_catalog), marker='P', color='royalblue', label='PN', size=34, zorder=29),
    ]:
        if handle is not None:
            compact_handles.append(handle)

    ax.coords[0].set_axislabel('RA')
    ax.coords[1].set_axislabel('Dec')
    if category_mode == 'snr_only':
        title = f'{mode}: S/N>3 all BPT lines black, low S/N grey; low S/N={int(df["low_snr_any_bpt_line"].sum())}'
        filename = f'{mode}_mosaic_boundaries_snr3_black_low_snr_gray.png'
    elif category_mode == 'snr_bpt_sf':
        title = f'{mode}: BPT SF black, high-S/N non-SF blue, low S/N grey; low S/N={int(df["low_snr_any_bpt_line"].sum())}'
        filename = f'{mode}_mosaic_boundaries_bpt_sf_black_high_snr_non_sf_blue_low_snr_gray.png'
    elif category_mode == 'nodig_sf_not_dig_sf':
        title = f'{mode}: black outlines are BPT SF before DIG correction but not after; N={int(df["nodig_sf_not_dig_sf_high_snr"].sum())}'
        filename = f'{mode}_mosaic_boundaries_nodig_sf_not_dig_sf_black.png'
    else:
        raise ValueError(category_mode)
    ax.set_title(title, fontsize=13)
    boundary_handles = [Line2D([0], [0], color=s['color'], lw=max(s['lw'] * 3, 1.8), label=s['label']) for s in styles.values()]
    ax.legend(handles=boundary_handles + compact_handles, frameon=True, fontsize=10, loc='lower left')
    out = mode_output_dir(mode, 'mosaic') / filename
    fig.savefig(out, dpi=350, bbox_inches='tight')
    plt.close(fig)
    return out


In [13]:
# Make all requested paired BPT-property plots.
property_keys = ['sigma', 'halpha_snr', 'region_size', 'sii_ha']
written = []
for mode, df in catalogs.items():
    for color_key in property_keys:
        written.append(plot_bpt_property_pair(df, mode, color_key))
    written.append(plot_bpt_category_pair(df, mode))
    if mode == 'dig_subtracted':
        written.append(plot_bpt_nodig_sf_highlight_pair(df, mode=mode))

print('Wrote BPT plots:')
for path in written:
    print(f'  {path}')


Wrote BPT plots:
  bpt_comparison/dig_subtracted/bpt/dig_subtracted_bpt_nii_sii_colored_by_sigma.png
  bpt_comparison/dig_subtracted/bpt/dig_subtracted_bpt_nii_sii_colored_by_halpha_snr.png
  bpt_comparison/dig_subtracted/bpt/dig_subtracted_bpt_nii_sii_colored_by_region_size.png
  bpt_comparison/dig_subtracted/bpt/dig_subtracted_bpt_nii_sii_colored_by_sii_ha.png
  bpt_comparison/dig_subtracted/bpt/dig_subtracted_bpt_nii_sii_snr_bptsf_categories.png
  bpt_comparison/dig_subtracted/bpt/dig_subtracted_bpt_nii_sii_nodig_sf_not_dig_sf_black_outline.png
  bpt_comparison/no_dig/bpt/no_dig_bpt_nii_sii_colored_by_sigma.png
  bpt_comparison/no_dig/bpt/no_dig_bpt_nii_sii_colored_by_halpha_snr.png
  bpt_comparison/no_dig/bpt/no_dig_bpt_nii_sii_colored_by_region_size.png
  bpt_comparison/no_dig/bpt/no_dig_bpt_nii_sii_colored_by_sii_ha.png
  bpt_comparison/no_dig/bpt/no_dig_bpt_nii_sii_snr_bptsf_categories.png


In [14]:
# Make requested boundary mosaics for both DIG modes.
written_mosaics = []
for mode, df in catalogs.items():
    written_mosaics.append(plot_boundary_mosaic(df, mode, 'snr_only'))
    written_mosaics.append(plot_boundary_mosaic(df, mode, 'snr_bpt_sf'))
    if mode == 'dig_subtracted':
        written_mosaics.append(plot_boundary_mosaic(df, mode, 'nodig_sf_not_dig_sf'))

print('Wrote mosaic plots:')
for path in written_mosaics:
    print(f'  {path}')


Wrote mosaic plots:
  bpt_comparison/dig_subtracted/mosaic/dig_subtracted_mosaic_boundaries_snr3_black_low_snr_gray.png
  bpt_comparison/dig_subtracted/mosaic/dig_subtracted_mosaic_boundaries_bpt_sf_black_high_snr_non_sf_blue_low_snr_gray.png
  bpt_comparison/dig_subtracted/mosaic/dig_subtracted_mosaic_boundaries_nodig_sf_not_dig_sf_black.png
  bpt_comparison/no_dig/mosaic/no_dig_mosaic_boundaries_snr3_black_low_snr_gray.png
  bpt_comparison/no_dig/mosaic/no_dig_mosaic_boundaries_bpt_sf_black_high_snr_non_sf_blue_low_snr_gray.png


## Output Summary

Each DIG mode writes to a separate folder:

- `bpt_comparison/dig_subtracted/bpt/`
- `bpt_comparison/dig_subtracted/mosaic/`
- `bpt_comparison/no_dig/bpt/`
- `bpt_comparison/no_dig/mosaic/`

The before/after DIG comparison cells write to:

- `bpt_comparison/comparison/bpt/`
- `bpt_comparison/comparison/luminosity_function/`
- `bpt_comparison/comparison/metallicity/`
- `bpt_comparison/comparison/tables/`
